In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

# Wczytaj dane z pliku .env
load_dotenv()

# Pobierz dane uwierzytelniające z pliku .env
CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

if not CLIENT_ID or not CLIENT_SECRET:
    print("Brak wymaganych danych SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET lub SPOTIFY_PLAYLIST_ID w pliku .env.")
    exit()

# Funkcja do uzyskania tokenu dostępu
def get_access_token(client_id, client_secret):
    try:
        url = "https://accounts.spotify.com/api/token"
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        data = {"grant_type": "client_credentials"}

        response = requests.post(url, headers=headers, data=data, auth=(client_id, client_secret))
        response.raise_for_status()

        token_data = response.json()
        return token_data.get("access_token")
    except Exception as e:
        print("Błąd podczas uzyskiwania tokenu dostępu:", str(e))
        exit()

# Funkcja do pobrania piosenek z playlisty z określonymi polami
def fetch_tracks_from_playlist(playlist_id, token):
    tracks = []
    try:
        url = f"https://api.spotify.com/v1/playlists/{playlist_id}/tracks"
        headers = {"Authorization": f"Bearer {token}"}
        params = {"limit": 100}  # Maksymalna liczba utworów w jednym żądaniu Spotify API

        while url:
            response = requests.get(url, headers=headers, params=params)
            response.raise_for_status()
            results = response.json()
            # Przetwarzanie danych
            for item in results.get("items", []):
                track = item.get("track")
                if track:
                    track_name = track.get("name", "Unknown")
                    track_url = track.get("external_urls", {}).get("spotify", "N/A")
                    album_name = track.get("album", {}).get("name", "Unknown")
                    artists = ", ".join(artist.get("name", "Unknown") for artist in track.get("artists", []))
                    tracks.append({
                        "Title": track_name,
                        "Artists": artists,
                        "Album": album_name,
                        "Spotify URL": track_url
                    })

            # Kontynuuj do następnej strony (paginacja)
            url = results.get("next")
            print(f"Pobrano {len(tracks)} utworów...")

    except Exception as e:
        print("Błąd podczas pobierania danych z playlisty:", str(e))

    return tracks

# Funkcja do zapisania danych do pliku CSV
def save_to_csv(tracks, filename="spotify_tracks.csv"):
    if not tracks:
        print("Brak danych do zapisania!")
        return

    try:
        df = pd.DataFrame(tracks)
        df.to_csv(filename, index=False, encoding="utf-8")
        print(f"Dane zapisane do pliku: {filename}")
    except Exception as e:
        print("Błąd podczas zapisywania danych do pliku CSV:", str(e))

# Pobierz token dostępu
access_token = get_access_token(CLIENT_ID, CLIENT_SECRET)
# Pobierz piosenki z playlisty

playlist_file = "playlists.txt"
if not os.path.exists(playlist_file):
    print(f"Plik {playlist_file} nie istnieje.")
    exit()

all_tracks = []

with open(playlist_file, "r") as file:
    for line in file:
        playlist_id = line.strip()
        if playlist_id:
            print(f"Przetwarzanie playlisty: {playlist_id}")
            tracks = fetch_tracks_from_playlist(playlist_id, access_token)
            all_tracks.extend(tracks)

# Zapisz dane do pliku CSV
save_to_csv(all_tracks)
